# 02. Modeling Preprocessing and Feature Sets

This notebook prepares the supervised modeling foundation for the dropout determinants study.

Purpose:
1. Load and lightly clean the dataset without changing the source CSV.
2. Define multiclass and binary dropout targets.
3. Define timing-based feature groups for staged early warning models.
4. Define variable blocks for ablation analysis.
5. Create one reproducible stratified train/validation/test split.
6. Save split summaries, feature metadata, and preprocessing metadata under `results/2.Model Processing/`.
7. Use one common encoded/scaled input representation for fair model comparison.

Important note: graph and risk-clustering analysis will later use the whole dataset. The train/validation/test split created here is only for supervised prediction models.

## 1. Imports, Paths, and Reproducibility

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

RANDOM_STATE = 123

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
PREP_DIR = RESULTS_DIR / "2.Model Processing"
PREP_CSV_DIR = PREP_DIR / "csv"
PREP_METADATA_DIR = PREP_DIR / "metadata"

PREP_CSV_DIR.mkdir(parents=True, exist_ok=True)
PREP_METADATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "data.csv"
DATA_PATH

## 2. Load Dataset and Clean Column Names

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")

# Keep source CSV unchanged; clean only the in-memory column labels.
df.columns = df.columns.str.strip()
df.insert(0, "StudentID", np.arange(len(df)))

audit_summary = pd.DataFrame({
    "metric": ["rows", "columns_including_student_id", "missing_values_total", "duplicate_rows"],
    "value": [len(df), df.shape[1], int(df.isna().sum().sum()), int(df.duplicated().sum())],
})

audit_summary.to_csv(PREP_CSV_DIR / "modeling_dataset_audit_summary.csv", index=False)
display(audit_summary)
df.head()

## 3. Target Definitions

In [ ]:
target_multiclass = "Target"
target_binary = "DropoutBinary"
target_binary_label = "DropoutLabel"

df[target_binary] = (df[target_multiclass] == "Dropout").astype(int)
df[target_binary_label] = np.where(df[target_binary] == 1, "Dropout", "NonDropout")

target_summary = pd.concat(
    [
        df[target_multiclass].value_counts().rename_axis("class").reset_index(name="count").assign(target="multiclass"),
        df[target_binary_label].value_counts().rename_axis("class").reset_index(name="count").assign(target="binary"),
    ],
    ignore_index=True,
)
target_summary["percent"] = target_summary.groupby("target")["count"].transform(lambda s: s / s.sum())
target_summary.to_csv(PREP_CSV_DIR / "target_definitions_summary.csv", index=False)
target_summary

## 4. Timing-Based Feature Sets for Early Warning

In [ ]:
pre_enrollment_features = [
    "Marital status",
    "Application mode",
    "Application order",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Previous qualification (grade)",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Admission grade",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "Age at enrollment",
    "International",
    "Unemployment rate",
    "Inflation rate",
    "GDP",
]

first_semester_features = [
    "Curricular units 1st sem (credited)",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 1st sem (without evaluations)",
]

second_semester_features = [
    "Curricular units 2nd sem (credited)",
    "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
    "Curricular units 2nd sem (without evaluations)",
]

stage_feature_sets = {
    "A_pre_enrollment": pre_enrollment_features,
    "B_first_semester": pre_enrollment_features + first_semester_features,
    "C_second_semester": pre_enrollment_features + first_semester_features + second_semester_features,
}

pd.DataFrame(
    [{"stage": name, "n_features": len(features)} for name, features in stage_feature_sets.items()]
)

## 5. Variable Blocks for Ablation Analysis

In [ ]:
academic_features = [
    "Previous qualification (grade)",
    "Admission grade",
] + first_semester_features + second_semester_features

financial_demographic_features = [
    "Age at enrollment",
    "Gender",
    "Marital status",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Scholarship holder",
    "International",
    "Nacionality",
]

parental_features = [
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
]

macro_features = ["Unemployment rate", "Inflation rate", "GDP"]
course_application_features = ["Course", "Application mode", "Application order"]

ablation_feature_sets = {
    "academic_only": academic_features,
    "academic_plus_financial_demographic": sorted(set(academic_features + financial_demographic_features)),
    "academic_plus_parental": sorted(set(academic_features + parental_features)),
    "academic_plus_macro": sorted(set(academic_features + macro_features)),
    "academic_plus_course_application": sorted(set(academic_features + course_application_features)),
    "full": stage_feature_sets["C_second_semester"],
}

pd.DataFrame(
    [{"feature_set": name, "n_features": len(features)} for name, features in ablation_feature_sets.items()]
)

## 6. Feature Validation and Type Metadata

In [ ]:
all_defined_features = sorted(
    set(
        pre_enrollment_features
        + first_semester_features
        + second_semester_features
        + academic_features
        + financial_demographic_features
        + parental_features
        + macro_features
        + course_application_features
    )
)

missing_features = sorted(set(all_defined_features) - set(df.columns))
unexpected_target_leakage = sorted({"Target", "DropoutBinary", "DropoutLabel", "StudentID"}.intersection(all_defined_features))

if missing_features:
    raise ValueError(f"Missing feature columns: {missing_features}")
if unexpected_target_leakage:
    raise ValueError(f"Target/id leakage in feature sets: {unexpected_target_leakage}")

categorical_features = [
    "Marital status",
    "Application mode",
    "Application order",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International",
]

numeric_features = [feature for feature in all_defined_features if feature not in categorical_features]

feature_metadata = pd.DataFrame(
    [
        {
            "feature": feature,
            "type_for_modeling": "categorical" if feature in categorical_features else "numeric",
            "dtype_in_csv": str(df[feature].dtype),
            "unique_count": int(df[feature].nunique(dropna=False)),
            "used_in_stage_A": feature in stage_feature_sets["A_pre_enrollment"],
            "used_in_stage_B": feature in stage_feature_sets["B_first_semester"],
            "used_in_stage_C": feature in stage_feature_sets["C_second_semester"],
        }
        for feature in all_defined_features
    ]
)

feature_metadata.to_csv(PREP_CSV_DIR / "feature_metadata.csv", index=False)
feature_metadata.head(20)

## 7. Stratified Train/Validation/Test Split

In [ ]:
def make_stratified_split(data, target_col, train_size=0.70, validation_size=0.20, random_state=123):
    rng = np.random.default_rng(random_state)
    split_labels = pd.Series(index=data.index, dtype="object")

    for _, group in data.groupby(target_col, sort=True):
        indices = group.index.to_numpy().copy()
        rng.shuffle(indices)

        n = len(indices)
        n_train = int(round(n * train_size))
        n_validation = int(round(n * validation_size))

        train_idx = indices[:n_train]
        validation_idx = indices[n_train:n_train + n_validation]
        test_idx = indices[n_train + n_validation:]

        split_labels.loc[train_idx] = "train"
        split_labels.loc[validation_idx] = "validation"
        split_labels.loc[test_idx] = "test"

    return split_labels

df["Split"] = make_stratified_split(df, target_multiclass, random_state=RANDOM_STATE)

split_indices = df[["StudentID", "Split", target_multiclass, target_binary, target_binary_label]].copy()
split_indices.to_csv(PREP_CSV_DIR / "train_validation_test_split_indices.csv", index=False)

split_indices["Split"].value_counts().rename_axis("split").reset_index(name="n")

## 8. Split Quality Checks

In [ ]:
split_multiclass_summary = (
    df.groupby(["Split", target_multiclass])
    .size()
    .rename("count")
    .reset_index()
)
split_multiclass_summary["percent_within_split"] = split_multiclass_summary.groupby("Split")["count"].transform(lambda s: s / s.sum())

split_binary_summary = (
    df.groupby(["Split", target_binary_label])
    .size()
    .rename("count")
    .reset_index()
)
split_binary_summary["percent_within_split"] = split_binary_summary.groupby("Split")["count"].transform(lambda s: s / s.sum())

split_multiclass_summary.to_csv(PREP_CSV_DIR / "split_multiclass_target_distribution.csv", index=False)
split_binary_summary.to_csv(PREP_CSV_DIR / "split_binary_target_distribution.csv", index=False)

display(split_multiclass_summary)
display(split_binary_summary)

## 9. Preprocessing Strategy for Later Model Notebooks

In [ ]:
preprocessing_strategy = {
    "main_fair_comparison": {
        "models": [
            "Decision Tree", "Random Forest", "XGBoost", "LightGBM",
            "CatBoost", "RBF SVM", "Neural Network"
        ],
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "encoding": "one-hot encoding for categorical features",
        "scaling": "standardize numeric features using training-set parameters only",
        "reason": "all models receive the same transformed input matrix for fair comparison",
    },
    "preprocessing_note": {
        "tree_based_models": "can work without scaling, but use the common encoded/scaled matrix here for direct comparison",
        "svm_and_neural_network": "require scaled numeric inputs and one-hot encoded categorical inputs",
        "leakage_control": "all encoding and scaling parameters must be fitted on training data only",
    },
    "catboost_sensitivity": {
        "recommended_encoding": "native categorical handling can be tested separately",
        "reason": "do not mix native CatBoost input with encoded/scaled input in the main fair-comparison table",
    },
}

with open(PREP_METADATA_DIR / "preprocessing_strategy.json", "w", encoding="utf-8") as f:
    json.dump(preprocessing_strategy, f, indent=2)

preprocessing_strategy.keys()

## 10. Save Feature-Set Metadata

In [ ]:
feature_sets_metadata = {
    "random_state": RANDOM_STATE,
    "targets": {
        "multiclass": target_multiclass,
        "binary_numeric": target_binary,
        "binary_label": target_binary_label,
    },
    "stage_feature_sets": stage_feature_sets,
    "ablation_feature_sets": ablation_feature_sets,
    "feature_types": {
        "categorical_features": categorical_features,
        "numeric_features": numeric_features,
    },
    "split_column": "Split",
    "id_column": "StudentID",
}

with open(PREP_METADATA_DIR / "feature_sets_metadata.json", "w", encoding="utf-8") as f:
    json.dump(feature_sets_metadata, f, indent=2)

feature_set_rows = []
for group_name, feature_set in {**stage_feature_sets, **ablation_feature_sets}.items():
    for feature in feature_set:
        feature_set_rows.append({"feature_set": group_name, "feature": feature})

feature_set_membership = pd.DataFrame(feature_set_rows)
feature_set_membership.to_csv(PREP_CSV_DIR / "feature_set_membership.csv", index=False)
feature_set_membership.head(20)

## 11. Create Modeling Master Dataset

In [ ]:
modeling_columns = [
    "StudentID",
    "Split",
    target_multiclass,
    target_binary,
    target_binary_label,
] + all_defined_features

modeling_df = df[modeling_columns].copy()
modeling_df.to_csv(PREP_CSV_DIR / "modeling_master_dataset_with_splits.csv", index=False)

modeling_df.shape

## 12. Status and Next Notebook

This notebook prepares the supervised modeling foundation. The main benchmark should use one common transformed input matrix for all models: one-hot encoded categorical variables and scaled numeric variables, fitted on training data only. This keeps the later staged model comparisons fair and avoids data leakage. The next notebook should be:

`3_SEW_Modeling.ipynb`

That notebook should load `results/2.Model Processing/csv/modeling_master_dataset_with_splits.csv` and `results/2.Model Processing/metadata/feature_sets_metadata.json`, then train models for Stage A, Stage B, and Stage C using the same split.